# Model Training

Train the prepared VGG16 transfer-learning model.

In [1]:

import os
from pathlib import Path

%pwd

'd:\\Projects\\1-Practice and Learning\\0 - COMPLETE NEW LEARNING ML and DL\\3-PROJECTS\\Deep Learning Projects\\Chest Cancer Classification End to End Project\\research'

In [2]:
os.chdir("../")

%pwd


'd:\\Projects\\1-Practice and Learning\\0 - COMPLETE NEW LEARNING ML and DL\\3-PROJECTS\\Deep Learning Projects\\Chest Cancer Classification End to End Project'

## 1. entity

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_learning_rate: float

## 2. Configuration manager

In [5]:
import tensorflow as tf

from cnnClassifier.constants import *
from cnnClassifier.utils.common import create_directories, read_yaml

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        training_data = Path(
            self.config.data_ingestion.unzip_dir
        ) / "CT Scan Dataset for Project" / "train"

        create_directories([training.root_dir])

        return TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=training_data,
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate=params.LEARNING_RATE
        )

## 3. Training component

In [8]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        # Do not restore the serialized optimizer. Keras 3 optimizers are tied
        # to the exact variable objects with which they were created.
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.config.params_learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.2,
                patience=2,
                min_lr=1e-7
            )
        ]

        self.history = self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            validation_data=self.valid_generator,
            callbacks=callbacks
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

        return self.history

## 4. Run training pipeline

In [9]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    history = training.train()
except Exception as e:
    raise e

[2026-09-22 13:33:50,864: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-09-22 13:33:50,868: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-09-22 13:33:50,869: INFO: common: Created directory at: artifacts]
[2026-09-22 13:33:50,871: INFO: common: Created directory at: artifacts/training]
[2026-09-22 13:33:51,253: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
Found 73 images belonging to 2 classes.
Found 296 images belonging to 2 classes.
Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 56s 3s/step - accuracy: 0.6318 - loss: 4.3786 - val_accuracy: 0.7534 - val_loss: 1.1827 - learning_rate: 1.0000e-04
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 66s 3s/step - accuracy: 0.8209 - loss: 1.4576 - val_accuracy: 0.8904 - val_loss: 0.7105 - learning_rate: 1.0000e-04
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 

## 5. Make .py files